In [1]:
pip install --quiet "numpy<2" opencv-python torch torchvision captum umap-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import argparse
import torch
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist
import seaborn as sns

sys.path.insert(1, '../')
import helpers
import umap

In [3]:
def load_checkpoint(checkpoint_path, model, device):
    """Load model weights from checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return checkpoint['epoch'], checkpoint.get('eval_metrics', {})


def collect_embeddings(model, dataloader, device, max_samples=2000):
    """Collect embeddings and labels from a dataloader"""
    model.eval()
    
    all_embeddings_l = []
    all_embeddings_r = []
    all_labels = []
    
    total_collected = 0
    
    with torch.no_grad():
        for _, (img1, img2, labels, _, _) in enumerate(dataloader):
            if total_collected >= max_samples:
                break
                
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            emb1, emb2 = model(img1, img2)
            
            all_embeddings_l.append(emb1.cpu().numpy())
            all_embeddings_r.append(emb2.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
            total_collected += len(labels)
    
    embeddings_l = np.vstack(all_embeddings_l)
    embeddings_r = np.vstack(all_embeddings_r)
    labels = np.hstack(all_labels)
    
    # Combine left and right
    all_embeddings = np.vstack([embeddings_l, embeddings_r])
    all_labels_combined = np.hstack([labels, labels])
    
    # Subsample if still too many
    if len(all_embeddings) > max_samples:
        indices = np.random.choice(len(all_embeddings), max_samples, replace=False)
        all_embeddings = all_embeddings[indices]
        all_labels_combined = all_labels_combined[indices]
    
    return all_embeddings, all_labels_combined


def compute_embedding_health(embeddings):
    """
    Compute metrics to detect embedding collapse
    """
    # 1. Effective rank
    try:
        S = np.linalg.svd(embeddings, compute_uv=False)
        S_normalized = S / (S.sum() + 1e-10)
        entropy = -np.sum(S_normalized * np.log(S_normalized + 1e-10))
        effective_rank = np.exp(entropy)
    except:
        effective_rank = -1
    
    # 2. Pairwise distance distribution
    sample_size = min(1000, len(embeddings))
    sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)
    distances = pdist(embeddings[sample_indices])
    distance_mean = np.mean(distances)
    distance_std = np.std(distances)
    
    # 3. Norm variance
    norms = np.linalg.norm(embeddings, axis=1)
    norm_mean = np.mean(norms)
    norm_std = np.std(norms)
    
    return {
        'effective_rank': effective_rank,
        'distance_mean': distance_mean,
        'distance_std': distance_std,
        'norm_mean': norm_mean,
        'norm_std': norm_std
    }


def compute_attention_entropy(model, dataloader, device, max_batches=10):
    """
    Compute attention entropy to see if attention is focused or uniform
    Only works for CrossAttentionSiamese models
    """
    if not hasattr(model, 'get_attention_maps'):
        return {'attention_entropy': None, 'normalized_entropy': None}
    
    model.eval()
    entropies = []
    
    with torch.no_grad():
        for batch_idx, (img1, img2, labels, _, _) in enumerate(dataloader):
            if batch_idx >= max_batches:
                break
                
            img1, img2 = img1.to(device), img2.to(device)
            
            try:
                # Get attention maps
                attn_weights = model.get_attention_maps(img1, img2, layer_idx=0)
                # attn_weights: (B, H, W, H, W)
                
                B, H, W, _, _ = attn_weights.shape
                attn_flat = attn_weights.view(B, H*W, H*W)
                
                # Compute entropy for each query position
                attn_probs = attn_flat + 1e-10
                entropy = -(attn_probs * torch.log(attn_probs)).sum(dim=-1)
                entropies.append(entropy.mean().cpu().item())
            except Exception as e:
                print(f"Warning: Could not compute attention entropy: {e}")
                return {'attention_entropy': None, 'normalized_entropy': None}
    
    if entropies:
        avg_entropy = np.mean(entropies)
        max_entropy = np.log(H * W)
        normalized_entropy = avg_entropy / max_entropy
        
        return {
            'attention_entropy': avg_entropy,
            'normalized_entropy': normalized_entropy
        }
    
    return {'attention_entropy': None, 'normalized_entropy': None}

def generate_multi_projection_plot(embeddings, labels, epoch, save_path, title_prefix=""):
    """
    Generate PCA, UMAP, and t-SNE visualizations side-by-side
    """
    projections = {}
    titles = []
    
    # 1. PCA 
    pca = PCA(n_components=2, random_state=42)
    projections['PCA'] = pca.fit_transform(embeddings)
    titles.append('PCA')
    # 2. UMAP
    try:
        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
        projections['UMAP'] = reducer.fit_transform(embeddings)
        titles.append('UMAP')
    except Exception as e:
        print(f"UMAP failed: {e}")

    # 3. t-SNE
    # try:
    #     tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)//4))
    #     projections['t-SNE'] = tsne.fit_transform(embeddings)
    #     titles.append('t-SNE')
    # except Exception as e:
    #     print(f"t-SNE failed: {e}")

    # Plot
    n_plots = len(projections)
    fig, axes = plt.subplots(1, n_plots, figsize=(7*n_plots, 6))
    if n_plots == 1:
        axes = [axes]
    
    for ax, method in zip(axes, titles):
        proj = projections[method]
        normal_mask = labels == 0
        nodule_mask = labels == 1
        
        ax.scatter(proj[normal_mask, 0], proj[normal_mask, 1], 
                  c='blue', alpha=0.6, label='Normal', s=20)
        ax.scatter(proj[nodule_mask, 0], proj[nodule_mask, 1], 
                  c='red', alpha=0.6, label='Nodule', s=20)
        ax.set_title(f'{method} Projection')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        if method == 'PCA':
            ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
            ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    
    fig.suptitle(f'{title_prefix}Embedding Projections (Epoch {epoch})', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close('all')
    print(f"  Saved projection plot: {save_path}")

def visualize_attention_maps(model, dataloader, device, save_path):
    """
    Visualize attention maps for a few samples
    Only works for CrossAttentionSiamese models
    """
    if not hasattr(model, 'get_attention_maps'):
        print("  Model doesn't have attention maps, skipping...")
        return
    
    model.eval()
    
    # Get one batch
    for img1, img2, labels, _, _ in dataloader:
        img1, img2 = img1.to(device), img2.to(device)
        
        try:
            with torch.no_grad():
                attn_weights = model.get_attention_maps(img1[:1], img2[:1], layer_idx=0)
            
            attn_weights = attn_weights.squeeze(0).cpu().numpy()  # (H, W, H, W)
            H, W = attn_weights.shape[:2]
            
            # Show attention from 4 key query positions
            fig, axes = plt.subplots(2, 2, figsize=(12, 12))
            positions = [(0, 0), (0, W-1), (H-1, 0), (H-1, W-1)]
            
            for ax, (h, w) in zip(axes.flat, positions):
                attn_map = attn_weights[h, w].reshape(H, W)
                im = ax.imshow(attn_map, cmap='hot', interpolation='nearest')
                ax.set_title(f'Attention from L[{h},{w}] to all R positions')
                plt.colorbar(im, ax=ax)
            
            plt.tight_layout()
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close('all')
            print(f"  Saved attention visualization: {save_path}")
            break
        except Exception as e:
            print(f"  Could not visualize attention: {e}")
            break


def analyze_checkpoint(checkpoint_path, model, dataloader, device, output_dir, dataset_name):
    """
    Analyze a single checkpoint
    """
    print(f"\nAnalyzing checkpoint: {checkpoint_path}")
    
    # Load checkpoint
    epoch, old_metrics = load_checkpoint(checkpoint_path, model, device)
    print(f"  Epoch: {epoch}")
    
    # Collect embeddings
    print("  Collecting embeddings...")
    embeddings, labels = collect_embeddings(model, dataloader, device)
    
    # Compute new metrics
    print("  Computing embedding health metrics...")
    health_metrics = compute_embedding_health(embeddings)
    
    print("  Computing attention metrics...")
    attention_metrics = compute_attention_entropy(model, dataloader, device)
    
    # Generate visualizations
    print("  Generating projection plots...")
    proj_save_path = os.path.join(output_dir, f'{dataset_name}_projections_epoch_{epoch}.png')
    generate_multi_projection_plot(embeddings, labels, epoch, proj_save_path, f"{dataset_name} - ")
    
    print("  Generating attention visualizations...")
    attn_save_path = os.path.join(output_dir, f'{dataset_name}_attention_epoch_{epoch}.png')
    visualize_attention_maps(model, dataloader, device, attn_save_path)
    
    # Combine all metrics
    all_metrics = {
        'epoch': epoch,
        **old_metrics,
        **health_metrics,
        **attention_metrics
    }
    
    return all_metrics




In [47]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [53]:
models_path = "logs/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/"
train_set = models_path.split("/")[2]
model_type = models_path.split("/")[3]
image_process = models_path.split("/")[4]
run_name = models_path.split("/")[5]
model_name = run_name.split("_")[0]
if model_type == "cross_attention":
    weights = run_name.split("_")[0]
    num_attn_layers = run_name.split("_")[2].split("L")[0][-1:]
    num_heads = run_name.split("_")[2].split("L")[1][0]
    use_symm = True if run_name.split("_")[4] == "sym" else False
    run_id = models_path.strip("/")[-1:]
else:
    raise NotImplementedError(f"Model type \"{model_type}\" unsupported! Model Type must be: cross_attention ")
resize_dim = 224
output_dir = models_path.replace("logs","plots")

In [50]:
datasets = ["chestxray14", "padchest", "jsrt"]
train_data_path = f"../split_node21_sets/{image_process}/{train_set}/train"
test_data_path = f"../split_node21_sets/{image_process}/{train_set}/test"
datasets.remove(train_set)
test2_data_path = f"../split_node21_sets/{image_process}/{datasets[0]}/test"
test3_data_path = f"../split_node21_sets/{image_process}/{datasets[1]}/test"

In [61]:
# umap_dir = os.path.join(output_dir,"umap")
# pca_dir = os.path.join(output_dir,"pca")
# os.makedirs(umap_dir, exist_ok=True)
# os.makedirs(pca_dir, exist_ok=True)

In [48]:
from torchvision import transforms

print(f"Building {model_type} model...")
backbone = helpers.models.load_truncated_model(model_name)

if model_type == "siamese":
    model = helpers.models.SiameseNetwork(
        backbone,
        embedding_dim=128,
        freeze_backbone=True
    ).to(device)
elif args.model_type == "cross_attention":
    from helpers.crossattention import CrossAttentionSiamese
    model = CrossAttentionSiamese(
        backbone,
        embedding_dim=128,
        num_attn_layers=num_attn_layers,
        num_heads=num_heads,
        freeze_backbone=True
    ).to(device)

# Find checkpoints
checkpoint_dir = Path(os.path.join(models_path,"checkpoints"))
checkpoint_files = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pth"))
print(f"\nFound {len(checkpoint_files)} checkpoint(s) to analyze")

all_results = []
for ckpt_path in checkpoint_files:
    metrics = analyze_checkpoint(
        str(ckpt_path),
        model,
        dataloader,
        device,
        args.output_dir,
        args.dataset_name
    )
    all_results.append(metrics)

for dataset_path in [train_data_path, test_data_path, test2_data_path, test3_data_path]:
    print(f"Loading dataset from: {dataset_path}")
    if model_name == "single":
        transform = transforms.Compose([
            transforms.Grayscale(1),
            transforms.Resize((args.resize_dim, args.resize_dim)),
            transforms.ToTensor(),
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((args.resize_dim, args.resize_dim)),
            transforms.ToTensor(),
        ])
    dataloader = helpers.dataloading.load_image_pair_dataset(
        dataset_path=dataset_path,
        batch_size=128,
        crop_size=resize_dim,
        symmetrical_transforms=use_symm,
        class_to_idx={'nodule': 1, 'normal': 0},
        transform=transform,
        cache_in_ram=True,
        single=(model_name == "single"),
        num_workers=10
    )
    

NameError: name 'dataset_path' is not defined